In [ ]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio import windows
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import SegformerForSemanticSegmentation
from tqdm import tqdm


# надо поменять абсолютные пути на ссылки S3
# ============================ НАСТРОЙКИ ============================
PICS_DIR = r'D:\kanopus_ikutsk\pics' # ссылка на tif
MASK_DIR = r'D:\kanopus_ikutsk\mask\заболачивание_scn' # ссылка на geoJson

PATCH_SIZE = 512
STRIDE = 256
BATCH_SIZE = 8
NUM_EPOCHS = 50
early_stopping = 9
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {DEVICE}')



# ============================ ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ============================
def get_file_pairs(pics_dir, mask_dir):
    """Возвращает список пар (tif_path, geojson_path) по совпадающим базовым именам."""
    pairs = []
    for fname in os.listdir(pics_dir):
        if fname.lower().endswith(('.tif', '.tiff')):
            base = os.path.splitext(fname)[0]
            geojson_path = os.path.join(mask_dir, 'irkutsk_' + base + '.geojson')
            if os.path.exists(geojson_path):
                pairs.append((os.path.join(pics_dir, fname), geojson_path))
    return pairs

def normalize_image(img):
    """Min-max нормализация по каждому каналу в [0, 1]."""
    img = img.astype(np.float32)
    for c in range(img.shape[0]):
        min_val = img[c].min()
        max_val = img[c].max()
        if max_val - min_val > 1e-6:
            img[c] = (img[c] - min_val) / (max_val - min_val)
        else:
            img[c] = 0
    return img

def geojson_to_mask_for_window(geojson_path, transform, out_shape):
    """Растеризует полигоны из GeoJSON в маску заданного размера и transform."""
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        return np.zeros(out_shape, dtype=np.uint8)
    shapes = [(geom, 1) for geom in gdf.geometry]
    mask = rasterize(shapes, out_shape=out_shape, transform=transform,
                     fill=0, dtype='uint8')
    return mask

# ============================ ФУНКЦИЯ ГЕНЕРАЦИИ КООРДИНАТ ============================
def build_all_patch_coords(pairs, patch_size, stride, context=0):
    """Генерирует координаты центральной области тайла с учётом контекста."""
    coords = []
    for tif_path, geojson_path in pairs:
        with rasterio.open(tif_path) as src:
            width, height = src.width, src.height
        # Координаты центрального окна, чтобы полное окно (с контекстом) не выходило за границы
        for y in range(context, height - patch_size - context + 1, stride):
            for x in range(context, width - patch_size - context + 1, stride):
                coords.append((tif_path, geojson_path, x, y))
    return coords


# ============================ ФУНКЦИЯ ДЛЯ ВЫЧИСЛЕНИЯ ВЕСОВ КЛАССОВ И СЭМПЛЕРА ============================
def compute_class_weights_and_sampler(dataset, num_samples=None):
    """
    Вычисляет веса классов на основе train-подвыборки.
    Также возвращает WeightedRandomSampler, который oversampling'ит патчи с классом 1.
    """
    positive_ratios = []
    labels_present = []
    print("Сканирование train-патчей для вычисления статистики...")
    for i in tqdm(range(len(dataset))):
        _, mask, pos_ratio = dataset[i]  # здесь dataset должен быть с return_positive_info=True
        positive_ratios.append(pos_ratio)
        labels_present.append(1 if pos_ratio > 0.001 else 0)  # считаем, что патч "положительный", если содержит класс

    positive_ratios = np.array(positive_ratios)
    labels_present = np.array(labels_present)

    # Веса классов для CrossEntropyLoss
    sum_pos = positive_ratios.sum()
    sum_neg = (1 - positive_ratios).sum()
    weight_pos = 1.0 / (sum_pos + 1e-6)
    weight_neg = 1.0 / (sum_neg + 1e-6)

    # Нормализуем, чтобы средний вес = 1
    mean_weight = (weight_pos + weight_neg) / 2
    weight_pos /= mean_weight
    weight_neg /= mean_weight
    class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32)

    # Сэмплер: даём больший вес патчам, где есть класс 1
    sample_weights = np.where(labels_present == 1, 15.0, 1.0)
    sample_weights = sample_weights / sample_weights.sum()
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(dataset), replacement=True)

    return class_weights, sampler


# ============================ ДАТАСЕТ С ДИНАМИЧЕСКОЙ ЗАГРУЗКОЙ ============================
class OnTheFlyDataset(Dataset):
    """
    Загружает патчи с диска по координатам.
    Не хранит все данные в памяти.
    """
    def __init__(self, patch_coords, patch_size, transform=None, return_positive_info=False):
        self.patch_coords = patch_coords
        self.patch_size = patch_size
        self.transform = transform
        self.return_positive_info = return_positive_info

    def __len__(self):
        return len(self.patch_coords)

    def __getitem__(self, idx):
        tif_path, geojson_path, x, y = self.patch_coords[idx]

        # Открываем TIF и читаем только нужное окно
        with rasterio.open(tif_path) as src:
            window = windows.Window(x, y, self.patch_size, self.patch_size)
            img = src.read(window=window)
            # Вычисляем transform для окна
            win_transform = src.window_transform(window)

        # Растеризуем GeoJSON для этого окна
        mask = geojson_to_mask_for_window(geojson_path, win_transform,
                                          (self.patch_size, self.patch_size))

        # Нормализация
        img = normalize_image(img)

        # Преобразование в тензоры
        img = torch.tensor(img, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        if self.transform:
            img, mask = self.transform(img, mask)

        if self.return_positive_info:
            # Возвращаем также долю положительных пикселей (для сэмплирования)
            positive_ratio = (mask == 1).float().mean().item()
            return img, mask, positive_ratio
        return img, mask



# ============================ ПОДГОТОВКА КООРДИНАТ И РАЗДЕЛЕНИЕ ============================
pairs = get_file_pairs(PICS_DIR, MASK_DIR)
print(f'Найдено пар файлов: {len(pairs)}')

all_coords = build_all_patch_coords(pairs, PATCH_SIZE, STRIDE)
print(f'Всего возможных патчей: {len(all_coords)}')

# Разделяем координаты на train/val/test (60/20/20) на уровне патчей
train_coords, temp_coords = train_test_split(all_coords, test_size=0.4, random_state=42)
val_coords, test_coords = train_test_split(temp_coords, test_size=0.5, random_state=42)
print(f'Train патчей: {len(train_coords)}, Val: {len(val_coords)}, Test: {len(test_coords)}')

# Создаём датасеты
train_dataset_info = OnTheFlyDataset(train_coords, PATCH_SIZE, return_positive_info=True)
val_dataset = OnTheFlyDataset(val_coords, PATCH_SIZE)
test_dataset = OnTheFlyDataset(test_coords, PATCH_SIZE)

# Веса классов и сэмплер для train
class_weights, train_sampler = compute_class_weights_and_sampler(train_dataset_info)

# Обычный train_dataset без positive_info для использования в DataLoader
train_dataset = OnTheFlyDataset(train_coords, PATCH_SIZE)



# DataLoader'ы
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=0
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)




model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",  
    num_labels=2,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)
model.to(DEVICE)



# Ищем первый свёрточный слой с 3 входными каналами
first_conv = None
for module in model.modules():
    if isinstance(module, torch.nn.Conv2d) and module.in_channels == 3:
        first_conv = module
        break

if first_conv is None:
    raise ValueError("Не найден свёрточный слой с 3 входными каналами")

# Создаём новый слой с 4 каналами
new_conv = torch.nn.Conv2d(
    in_channels=4,
    out_channels=first_conv.out_channels,
    kernel_size=first_conv.kernel_size,
    stride=first_conv.stride,
    padding=first_conv.padding,
    bias=first_conv.bias is not None
)

# Копируем веса: первые 3 канала — RGB, 4-й канал дублируем (например, с красного)
with torch.no_grad():
    new_conv.weight[:, :3] = first_conv.weight
    new_conv.weight[:, 3] = first_conv.weight[:, 0]  # дублируем веса красного канала

# Заменяем найденный слой в модели
for name, module in model.named_modules():
    if module is first_conv:
        parent_name = name.rsplit('.', 1)[0] if '.' in name else ''
        parent = model.get_submodule(parent_name) if parent_name else model
        attr_name = name.rsplit('.', 1)[-1] if '.' in name else name
        setattr(parent, attr_name, new_conv)
        print(f"Заменён слой: {name}")
        break

# Обновляем конфигурацию модели
model.config.num_channels = 4

model.to(DEVICE)
print(f'Модель SegFormer-B0 загружена, число классов: {model.config.num_labels}, входных каналов: {model.config.num_channels}')



# ============================ LOSS, ОПТИМИЗАТОР ============================
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)



# ============================ ЦИКЛ ОБУЧЕНИЯ ============================
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    # Обучение
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]')
    for images, masks in pbar:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)   # выход: logits (B, C, H/4, W/4)
        logits = outputs.logits
        # Апсемплим до размера маски
        logits = torch.nn.functional.interpolate(
            logits,
            size=masks.shape[-2:],
            mode='bilinear',
            align_corners=False
        )
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({'loss': loss.item()})
    train_loss /= len(train_loader.dataset)

    # Валидация
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            logits = outputs.logits
            logits = torch.nn.functional.interpolate(
                logits,
                size=masks.shape[-2:],
                mode='bilinear',
                align_corners=False
            )
            loss = criterion(logits, masks)
            val_loss += loss.item() * images.size(0)
    val_loss /= len(val_loader.dataset)

    print(f'Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}')

    scheduler.step(val_loss)

    # Сохранение лучшей модели
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_segformer_заболачивание_main.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= early_stopping:
            print('Ранняя остановка.')
            break
